# Fabric stain tiled-inference validation

This is a **development/model-selection run**, not a final test. It compares the existing
whole-image resize with a tiled method that matches the model's 512-pixel training crop scale.

- Training remains self-supervised and uses normal fabric only.
- The 10 normal validation images calibrate false alarms.
- Thirty stain images that were not used in V1-V5 select the inference method and threshold.
- The original 100 test stains are excluded from this run.
- After this notebook, freeze the selected settings and run them once on a new final cohort.

Why tiling may help: the current evaluation compresses a 1984x1488 image to 224x224, which can
erase a small stain. Tiling crops the original at 512x512, as in training, before resizing each
crop to 224x224.


In [ ]:
from pathlib import Path
import csv, importlib.util, json, random, shutil, sys, time, zipfile

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score

assert Path('/kaggle/input').is_dir(), 'Run this notebook on Kaggle.'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'

KAGGLE_INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/labelinspect')
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT = WORK / 'fabric_stain_tiled_validation'
OUTPUT.mkdir(parents=True, exist_ok=True)

def find_dataset(search_root):
    found = []
    for path in search_root.rglob('fabric_stain_pilot'):
        if path.is_dir() and (path / 'val_normal').is_dir() and (path / 'test' / 'stain').is_dir():
            found.append(path)
    return sorted(set(found))

dataset_candidates = find_dataset(KAGGLE_INPUT)
if not dataset_candidates:
    matching = []
    for archive_path in KAGGLE_INPUT.rglob('*.zip'):
        with zipfile.ZipFile(archive_path) as archive:
            names = ['/' + item.filename.replace('\\', '/').lstrip('/') for item in archive.infolist()]
            if any('/fabric_stain_pilot/val_normal/' in name for name in names):
                matching.append(archive_path)
    if len(matching) == 1:
        extraction_root = WORK / 'uploaded_data'
        extraction_root.mkdir(parents=True, exist_ok=True)
        resolved = extraction_root.resolve()
        with zipfile.ZipFile(matching[0]) as archive:
            for item in archive.infolist():
                target = (extraction_root / item.filename).resolve()
                if target != resolved and resolved not in target.parents:
                    raise ValueError(f'Unsafe ZIP member: {item.filename}')
            archive.extractall(extraction_root)
        dataset_candidates = find_dataset(extraction_root)
if len(dataset_candidates) != 1:
    raise FileNotFoundError('Expected one fabric_stain_pilot dataset: ' + repr([str(p) for p in dataset_candidates]))
DATA_ROOT = dataset_candidates[0]

all_pt_files = sorted(KAGGLE_INPUT.rglob('*.pt'))
if not all_pt_files:
    extracted_roots = []
    for data_pickle in KAGGLE_INPUT.rglob('data.pkl'):
        candidate = data_pickle.parent
        if (candidate / 'data').is_dir() and (candidate / 'version').is_file():
            extracted_roots.append(candidate)
    extracted_roots = sorted(set(extracted_roots))
    if len(extracted_roots) == 1:
        archive_root = extracted_roots[0]
        rebuilt = WORK / 'rebuilt_fabric_stain_checkpoint.pt'
        with zipfile.ZipFile(rebuilt, 'w', compression=zipfile.ZIP_STORED) as archive:
            for source_file in sorted(archive_root.rglob('*')):
                if source_file.is_file():
                    archive.write(source_file, f'{archive_root.name}/{source_file.relative_to(archive_root).as_posix()}')
        all_pt_files = [rebuilt]
        print('Rebuilt Kaggle-extracted checkpoint:', rebuilt)
if len(all_pt_files) != 1:
    raise FileNotFoundError('Expected one checkpoint input; found: ' + repr([str(p) for p in all_pt_files]))
CHECKPOINT = all_pt_files[0]

SEED = 230224
IMAGE_SIZE = 224
T_DISTANCE = 50
NUM_DIFFUSION_SAMPLES = 1
TILE_SOURCE_SIZE = 512
TILE_STRIDE = 384
TILE_BATCH_SIZE = 12
VALIDATION_STAIN_COUNT = 30
DETREND_KERNEL_1 = 41
DETREND_KERNEL_2 = 81
SMOOTH_KERNEL = 5
TILE_PIXEL_QUANTILE = 0.999

print('GPU:', torch.cuda.get_device_name(0))
print('Dataset:', DATA_ROOT)
print('Checkpoint:', CHECKPOINT)


In [ ]:
script = WORK / 'author_smoke.py'
script.write_text('"""Run a bounded integration check of the author DTU-Net and Tsimplex code.\n\nThis is not training for anomaly detection, not a reproduced result, and not a\nperformance comparison. Downloads only five source files from a pinned commit.\n"""\nfrom __future__ import annotations\nimport argparse\nimport ast\nimport hashlib\nimport importlib.util\nimport json\nimport random\nimport sys\nimport time\nimport types\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\n\nCOMMIT = \'dc4a9bd2a2a5b1c31223daab4bdfea3f6a5b2990\'\nBASE_URL = f\'https://raw.githubusercontent.com/MAXNORM8650/Annotsim/{COMMIT}/\'\nFILES = [\'src/models/UModels/UDHVT.py\',\'GaussianDiffusion.py\',\n         \'utils/Simplex/constants.py\',\'utils/Simplex/internals.py\',\'utils/Simplex/noise.py\']\nEXPECTED = {\n \'utils/Simplex/constants.py\':\'52bf6ba3e2c0d386fa420382de380093a8dd61f488765cb812b13e25d0be7294\',\n \'utils/Simplex/internals.py\':\'ef67562885dcfe3356acd97784fe10660bf21238be7bcc608e86053c529fd61a\',\n \'src/models/UModels/UDHVT.py\':\'f7303c4dd228a3f5e1ab98d16fe1db7c7abfecfa97f683e89449128c5a03a4c2\',\n \'GaussianDiffusion.py\':\'cdf7a2143a441d20a3250458c0683c53ac1484ef8f0d0927831e66a34b52ec9a\',\n \'utils/Simplex/noise.py\':\'d114b6898369a0299e48f95fe4165fb3d587dcb8d7e257b58aef02077b6249d3\',\n}\n\n\ndef fetch_sources(root):\n    hashes={}\n    for name in FILES:\n        destination=root/name\n        destination.parent.mkdir(parents=True,exist_ok=True)\n        if not destination.exists():\n            print(\'Downloading\',name,flush=True)\n            last_error=None\n            for attempt in range(1,4):\n                try:\n                    with urllib.request.urlopen(BASE_URL+name,timeout=45) as response:\n                        payload=response.read()\n                    break\n                except urllib.error.URLError as exc:\n                    last_error=exc\n                    print(f\'Network attempt {attempt}/3 failed: {exc}\',flush=True)\n                    if attempt < 3:time.sleep(2*attempt)\n            else:\n                raise RuntimeError(\n                    \'Could not download the pinned public author files. In Kaggle, \'\n                    \'open Settings, turn Internet on, then rerun this cell.\'\n                ) from last_error\n            if name in EXPECTED and hashlib.sha256(payload).hexdigest()!=EXPECTED[name]:\n                raise ValueError(f\'Inspected-source hash mismatch: {name}; stop and review this revision.\')\n            destination.write_bytes(payload)\n        digest=hashlib.sha256(destination.read_bytes()).hexdigest()\n        if name in EXPECTED and digest!=EXPECTED[name]:\n            raise ValueError(f\'Cached-source hash mismatch: {name}; use a fresh cache after review.\')\n        hashes[name]=digest\n    return hashes\n\n\ndef load_module(name,path):\n    spec=importlib.util.spec_from_file_location(name,path)\n    module=importlib.util.module_from_spec(spec)\n    sys.modules[name]=module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef load_author_components(source_root,output):\n    import numpy as np\n    import torch\n    import torch.nn as nn\n    # Namespace isolation avoids importing the repository\'s unrelated experiments.\n    package=types.ModuleType(\'labelinspect_author_simplex\')\n    package.__path__=[str(source_root/\'utils/Simplex\')]\n    sys.modules[package.__name__]=package\n    noise=load_module(package.__name__+\'.noise\',source_root/\'utils/Simplex/noise.py\')\n\n    original=(source_root/\'src/models/UModels/UDHVT.py\').read_text(encoding=\'utf8\')\n    replacements={\n      \'from torchvision import models\':\'# Removed unused torchvision.models import.\',\n      \'from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD, IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD\':\'# Removed unused timm image constants.\',\n      \'from timm.models.helpers import build_model_with_cfg, named_apply, adapt_input_conv\':\'from timm.models._manipulate import named_apply\',\n      \'from timm.models.layers import trunc_normal_, lecun_normal_, to_2tuple\':\'from timm.layers import trunc_normal_, lecun_normal_, to_2tuple\',\n      \'from timm.models.registry import register_model\':\'# Removed unused timm registry import.\',\n    }\n    for old,new in replacements.items():\n        if original.count(old)!=1:raise ValueError(\'Compatibility patch no longer matches the inspected source: \'+old)\n        original=original.replace(old,new)\n    patched=output/\'author_UDHVT_compat.py\'\n    patched.write_text(\'import numpy as np\\n\'+original,encoding=\'utf8\')\n    model_module=load_module(\'labelinspect_author_model\',patched)\n\n    # Keep the author definitions, including its variance convention. Avoid the\n    # top-level imports for unused image losses, plotting, datasets and backbones.\n    tree=ast.parse((source_root/\'GaussianDiffusion.py\').read_text(encoding=\'utf8\'))\n    wanted={\'get_beta_schedule\',\'extract\',\'mean_flat\',\'generate_simplex_4noise\',\'GaussianDiffusionModel\'}\n    selected=[node for node in tree.body if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and node.name in wanted]\n    if {node.name for node in selected}!=wanted:raise ValueError(\'Required author diffusion definitions are missing\')\n    reduced=ast.Module(body=selected,type_ignores=[])\n    namespace={\'np\':np,\'torch\':torch,\'nn\':nn,\'OpenSimplex\':noise.OpenSimplex}\n    exec(compile(reduced,\'author_diffusion_l2_subset.py\',\'exec\'),namespace)\n    (output/\'author_diffusion_l2_subset.py\').write_text(ast.unparse(reduced),encoding=\'utf8\')\n\n    class NoisePredictionAdapter(nn.Module):\n        def __init__(self,backbone):super().__init__();self.backbone=backbone\n        def forward(self,x,t,y=None):\n            if y is not None:raise ValueError(\'This initial adapter supports the normal-only, unconditioned path\')\n            result=self.backbone(x,t,y=None)\n            prediction=result[0] if isinstance(result,tuple) else result\n            if prediction.shape!=x.shape:raise ValueError(\'Noise prediction does not match input shape\')\n            return prediction\n\n    return model_module,namespace,NoisePredictionAdapter,{\n      \'imports\':replacements,\'extra_import\':\'numpy for the author PositionalEmbedding helper\',\n      \'adapter\':\'Select tuple element 0; preserve the backbone computation.\',\n      \'diffusion_loading\':\'AST-load only the author definitions required for Gaussian/Tsimplex L2 and sampling; other losses are not supported.\',\n      \'sampling\':\'Pass denoise_fn=noise_fn so reverse steps use configured O/mu/p instead of the author alternate branch defaults.\',\n      \'calling_convention\':\'Set author diffusion train=False to select model(x,t,y=lab) during sampling. This is a dispatch flag; the model is explicitly switched with model.train()/eval().\',\n    }\n\n\ndef synthetic_batch(size,batch,device):\n    import numpy as np\n    import torch\n    from PIL import Image,ImageDraw,ImageFont\n    im=Image.new(\'L\',(size,size),235);draw=ImageDraw.Draw(im)\n    try:font=ImageFont.truetype(\'DejaVuSans.ttf\',20)\n    except OSError:font=ImageFont.load_default(size=20)\n    draw.rectangle((12,12,size-12,size-12),outline=20,width=2)\n    draw.text((24,45),\'LABEL A-104\',font=font,fill=20)\n    draw.text((24,90),\'BATCH 2026\',font=font,fill=20)\n    x=torch.from_numpy(np.asarray(im).copy()).float()/127.5-1\n    return x[None,None].repeat(batch,3,1,1).to(device)\n\n\ndef save_preview(x,reconstructed,destination):\n    from PIL import Image,ImageDraw\n    import numpy as np\n    images=[]\n    for tensor in [x,reconstructed]:\n        array=((tensor[0].detach().float().cpu().permute(1,2,0).numpy()+1)/2*255).clip(0,255).astype(np.uint8)\n        images.append(Image.fromarray(array))\n    sheet=Image.new(\'RGB\',(520,302),\'white\');draw=ImageDraw.Draw(sheet)\n    draw.text((12,10),\'INTEGRATION CHECK ONLY - TWO UPDATES\',fill=\'darkred\')\n    draw.text((12,32),\'Synthetic input\',fill=\'black\');draw.text((268,32),\'8-step reconstruction\',fill=\'black\')\n    for i,im in enumerate(images):sheet.paste(im.resize((224,224)),(12+i*256,54))\n    draw.text((12,283),\'No anomaly-removal or accuracy claim.\',fill=\'darkred\');sheet.save(destination)\n\n\ndef run(output,cache=None):\n    import importlib.metadata\n    import numpy as np\n    import torch\n    import numba\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if not torch.cuda.is_available():raise RuntimeError(\'Select a GPU accelerator before running this notebook\')\n    cache=Path(cache) if cache else output/\'upstream\'/COMMIT\n    hashes=fetch_sources(cache)\n    random.seed(230224);np.random.seed(230224);torch.manual_seed(230224)\n    numba.set_num_threads(min(2,numba.get_num_threads()))\n    module,ns,adapter_type,patches=load_author_components(cache,output)\n    config={\'img_size\':224,\'patch_size\':16,\'in_chans\':3,\'embed_dim\':384,\'depth\':12,\n            \'num_heads\':6,\'mlp_ratio\':4.,\'num_classes\':None,\'mlp_time_embed\':True,\n            \'use_dec\':[\'DAFF\',\'DAFF\',\'DAFF\'],\'PE_type\':\'SPE\',\'refinement\':True,\'qkv_bias\':False}\n    report={\'status\':\'running\',\'scope\':\'integration_check_only\',\'upstream_commit\':COMMIT,\n            \'source_sha256\':hashes,\'compatibility_changes\':patches,\'model_configuration\':config,\n            \'configuration_note\':\'Illustrated SPE/DMHA/HFF/refinement variant. Code depth=12 gives six encoder blocks, one middle, six decoder blocks. This is not asserted to match every paper table.\',\n            \'torch\':torch.__version__,\'gpu\':torch.cuda.get_device_name(0),\n            \'gpu_vram_gib\':torch.cuda.get_device_properties(0).total_memory/2**30,\n            \'packages\':{n:importlib.metadata.version(n) for n in [\'timm\',\'einops\',\'numba\',\'numpy\']},\n            \'noise_parameters\':{\'octave\':6,\'frequency\':64,\'persistence\':.9},\n            \'training_steps\':2,\'batch_size\':2,\'total_diffusion_steps\':1000,\'reconstruction_steps\':8}\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    device=torch.device(\'cuda:0\');torch.cuda.reset_peak_memory_stats()\n    print(\'Building author DTU-Net, width 384, six attention heads...\',flush=True)\n    backbone=module.UDHVT(**config).to(device);model=adapter_type(backbone)\n    report[\'parameter_count\']=sum(p.numel() for p in model.parameters())\n    x=synthetic_batch(224,2,device)\n    t=torch.tensor([50,150],device=device,dtype=torch.long)\n    diffusion=ns[\'GaussianDiffusionModel\']([224,224],ns[\'get_beta_schedule\'](1000,\'cosine\'),img_channels=3,\n                 loss_type=\'l2\',noise=\'4dsimplex\',octave=6,frequency=64,persistence=.9,train=False)\n    print(\'Compiling the author 4D noise function on CPU; first use may take a few minutes...\',flush=True)\n    start=time.perf_counter()\n    probe=torch.zeros(1,1,4,4,device=device)\n    diffusion.noise_fn(probe,torch.tensor([5],device=device))\n    report[\'noise_first_compile_seconds\']=time.perf_counter()-start\n    noise=diffusion.noise_fn(x,t).float()\n    assert noise.shape==x.shape and torch.isfinite(noise).all()\n    assert not torch.allclose(noise[0],noise[1]),\'Different time coordinates unexpectedly generated identical samples\'\n    report[\'noise_shape\']=list(noise.shape)\n    report[\'noise_mean\']=float(noise.mean());report[\'noise_std\']=float(noise.std())\n    report[\'noise_normalization\']=\'Author raw amplitude retained; no per-sample standardization.\'\n    report[\'batch_noise_note\']=\'The author generator uses t as the fourth coordinate. Duplicate time coordinates with one seed can produce identical noise across batch entries; this remains to be assessed during training.\'\n    model.train();optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=0.)\n    report[\'losses\']=[];report[\'gradient_norms\']=[]\n    tracked=backbone.pos_embed.detach().clone()\n    for step in range(2):\n        optimizer.zero_grad(set_to_none=True)\n        losses,noisy,predicted=diffusion.calc_loss(model,x,None,t)\n        loss=losses[\'loss\'].mean()\n        assert predicted.shape==x.shape and torch.isfinite(loss)\n        loss.backward()\n        grads=[p.grad for p in model.parameters() if p.grad is not None]\n        assert grads and all(torch.isfinite(g).all() for g in grads)\n        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n        optimizer.step()\n        report[\'losses\'].append(float(loss.detach()))\n        report[\'gradient_norms\'].append(float(norm))\n        print(f\'Update {step+1}/2 passed; L2 noise loss {float(loss.detach()):.6f}\',flush=True)\n    assert not torch.equal(tracked,backbone.pos_embed.detach()),\'Optimizer did not change the tracked parameter\'\n    report[\'tracked_parameter_changed\']=True\n    report[\'parameters_without_grad\']=[name for name,p in model.named_parameters() if p.grad is None]\n    model.eval()\n    print(\'Checking eight author reverse-diffusion steps. This model is not trained for detection.\',flush=True)\n    with torch.no_grad():\n        result=diffusion.forward_backward(model,x[:1],None,see_whole_sequence=None,t_distance=8,denoise_fn=\'noise_fn\')\n    assert result.shape==x[:1].shape and torch.isfinite(result).all()\n    residual=(x[:1]-result).square().mean(dim=1)\n    assert residual.shape==(1,224,224) and torch.isfinite(residual).all()\n    save_preview(x,result,output/\'integration_preview.png\')\n    report[\'reconstruction_shape\']=list(result.shape);report[\'residual_shape\']=list(residual.shape)\n    report[\'peak_gpu_allocated_gib\']=torch.cuda.max_memory_allocated()/2**30\n    report[\'peak_gpu_reserved_gib\']=torch.cuda.max_memory_reserved()/2**30\n    report[\'status\']=\'passed\'\n    report[\'not_completed\']=[\'Training a useful anomaly model\',\'Real label dataset\',\'Paper metrics reproduction\',\'Quality comparison with CPU baseline\']\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'\\nDTU-NET + TSIMPLEX INTEGRATION CHECK PASSED\',flush=True)\n    print(json.dumps({k:report[k] for k in [\'parameter_count\',\'noise_shape\',\'losses\',\'reconstruction_shape\',\'peak_gpu_allocated_gib\',\'peak_gpu_reserved_gib\']},indent=2))\n    print(\'Saved:\',output/\'integration_report.json\',flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\',default=\'artifacts/author_integration\')\n    parser.add_argument(\'--cache\',default=None)\n    args=parser.parse_args();run(args.output,args.cache)\n', encoding='utf8')
print('Wrote:', script)


In [ ]:
spec = importlib.util.spec_from_file_location('labelinspect_author_smoke', WORK / 'author_smoke.py')
author = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = author
spec.loader.exec_module(author)
import numba
numba.set_num_threads(min(2, numba.get_num_threads()))
source_cache = WORK / 'upstream' / author.COMMIT
hashes = author.fetch_sources(source_cache)
model_module, diffusion_ns, Adapter, compatibility = author.load_author_components(source_cache, OUTPUT)
print('Pinned author commit:', author.COMMIT)


In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
def image_paths(folder):
    return sorted(path for path in folder.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)

val_normal_paths = image_paths(DATA_ROOT / 'val_normal')
all_stain_paths = image_paths(DATA_ROOT / 'test' / 'stain')
assert len(val_normal_paths) == 10 and len(all_stain_paths) == 398

# V1-V5 used shuffled positions 0:100. These validation stains come from 100:130.
rng = random.Random(SEED)
shuffled_stains = all_stain_paths.copy()
rng.shuffle(shuffled_stains)
validation_stain_paths = sorted(shuffled_stains[100:100 + VALIDATION_STAIN_COUNT])
validation_paths = val_normal_paths + validation_stain_paths
labels = np.asarray([0] * len(val_normal_paths) + [1] * len(validation_stain_paths), dtype=np.int64)

(OUTPUT / 'development_validation_files.txt').write_text(
    '\n'.join(path.relative_to(DATA_ROOT).as_posix() for path in validation_paths)
)
print('Development set:', len(val_normal_paths), 'normal +', len(validation_stain_paths), 'previously unused stains')


In [ ]:
device = torch.device('cuda:0')
checkpoint = torch.load(CHECKPOINT, map_location=device, weights_only=False)
assert checkpoint['step'] == 2000, f"Expected step 2000, found {checkpoint['step']}"
assert checkpoint['author_commit'] == author.COMMIT
assert checkpoint.get('dataset_category') == 'fabric_stain_pilot'
model_config = checkpoint['model_config']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
backbone = model_module.UDHVT(**model_config).to(device)
model = Adapter(backbone)
model.load_state_dict(checkpoint['model'])
model.eval()
diffusion = diffusion_ns['GaussianDiffusionModel'](
    [224, 224], diffusion_ns['get_beta_schedule'](1000, 'cosine'), img_channels=3,
    loss_type='l2', noise='4dsimplex', octave=6, frequency=64, persistence=0.9, train=False)
diffusion.noise_fn(torch.zeros(1, 1, 4, 4, device=device), torch.tensor([5], device=device))
torch.cuda.reset_peak_memory_stats()
print('Loaded step', checkpoint['step'], 'model with', sum(p.numel() for p in model.parameters()), 'parameters.')


In [ ]:
RESAMPLE = getattr(Image, 'Resampling', Image).BILINEAR

def pil_to_tensor(image):
    image = image.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE)
    array = np.asarray(image, dtype=np.float32).copy() / 127.5 - 1.0
    return torch.from_numpy(array).permute(2, 0, 1)

def reconstruct_tensor_batch(batch):
    x = batch.to(device)
    with torch.inference_mode():
        samples = []
        for _ in range(NUM_DIFFUSION_SAMPLES):
            samples.append(diffusion.forward_backward(
                model, x, None, see_whole_sequence=None,
                t_distance=T_DISTANCE, denoise_fn='noise_fn'))
        recon = torch.stack(samples).mean(dim=0)
    return x.cpu(), recon.cpu()

def residual_maps(inputs, recons):
    weights = torch.tensor([0.2989, 0.5870, 0.1140]).view(1, 3, 1, 1)
    raw = ((inputs * weights).sum(1, keepdim=True) - (recons * weights).sum(1, keepdim=True)).square()

    def reflected_mean(x, kernel):
        pad = kernel // 2
        return F.avg_pool2d(F.pad(x, (pad, pad, pad, pad), mode='reflect'), kernel, stride=1)

    det1 = torch.clamp(raw - reflected_mean(raw, DETREND_KERNEL_1), min=0)
    det2 = torch.clamp(raw - reflected_mean(raw, DETREND_KERNEL_2), min=0)
    detrended = torch.maximum(det1, 0.9 * det2)
    smoothed = reflected_mean(detrended, SMOOTH_KERNEL).squeeze(1).numpy().astype(np.float32)

    flat = smoothed.reshape(len(smoothed), -1)
    med = np.median(flat, axis=1, keepdims=True)
    q25 = np.quantile(flat, 0.25, axis=1, keepdims=True)
    q75 = np.quantile(flat, 0.75, axis=1, keepdims=True)
    iqr = np.maximum(q75 - q25, 1e-6)
    return ((flat - med) / iqr).reshape(smoothed.shape)

def positions(length, crop=TILE_SOURCE_SIZE, stride=TILE_STRIDE):
    if length <= crop:
        return [0]
    values = list(range(0, length - crop + 1, stride))
    if values[-1] != length - crop:
        values.append(length - crop)
    return values

def evaluate_path(path):
    with Image.open(path) as source_image:
        full = source_image.convert('RGB')
        width, height = full.size

        global_tensor = pil_to_tensor(full).unsqueeze(0)
        global_input, global_recon = reconstruct_tensor_batch(global_tensor)
        global_map = residual_maps(global_input, global_recon)[0]
        global_score = float(np.max(global_map))

        tile_tensors = []
        tile_boxes = []
        for top in positions(height):
            for left in positions(width):
                box = (left, top, min(left + TILE_SOURCE_SIZE, width), min(top + TILE_SOURCE_SIZE, height))
                tile_tensors.append(pil_to_tensor(full.crop(box)))
                tile_boxes.append(box)

    tile_maps = []
    tick = time.perf_counter()
    for start in range(0, len(tile_tensors), TILE_BATCH_SIZE):
        batch = torch.stack(tile_tensors[start:start + TILE_BATCH_SIZE])
        tile_inputs, tile_recons = reconstruct_tensor_batch(batch)
        tile_maps.extend(residual_maps(tile_inputs, tile_recons))
    elapsed = time.perf_counter() - tick

    tile_maps = np.asarray(tile_maps, dtype=np.float32)
    tile_scores = np.quantile(tile_maps.reshape(len(tile_maps), -1), TILE_PIXEL_QUANTILE, axis=1)
    top_two = np.sort(tile_scores)[-min(2, len(tile_scores)):]
    return {
        'global_score': global_score,
        'tile_max_score': float(np.max(tile_scores)),
        'tile_top2_mean_score': float(np.mean(top_two)),
        'tile_count': len(tile_scores),
        'seconds': elapsed,
        'tile_maps': tile_maps,
        'tile_boxes': tile_boxes,
        'image_size': (width, height),
    }


In [ ]:
records = []
preview_cache = {}
run_tick = time.perf_counter()
for index, (path, label) in enumerate(zip(validation_paths, labels)):
    result = evaluate_path(path)
    records.append({
        'image': path.relative_to(DATA_ROOT).as_posix(),
        'label': int(label),
        'kind': 'stain' if label else 'good',
        'global_score': result['global_score'],
        'tile_max_score': result['tile_max_score'],
        'tile_top2_mean_score': result['tile_top2_mean_score'],
        'tile_count': result['tile_count'],
        'tile_seconds': result['seconds'],
    })
    if index in {0, 1, 10, 11, 12, 13}:
        preview_cache[index] = result
    print(f"{index + 1}/{len(validation_paths)} {path.name}: global={result['global_score']:.2f}, "
          f"tile-top2={result['tile_top2_mean_score']:.2f}, {result['tile_count']} tiles", flush=True)
run_seconds = time.perf_counter() - run_tick

with (OUTPUT / 'validation_per_image.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(records[0]))
    writer.writeheader(); writer.writerows(records)
print('Validation runtime (minutes):', run_seconds / 60)


In [ ]:
def choose_threshold(y, scores, minimum_specificity=0.90):
    candidates = np.r_[-np.inf, np.unique(scores), np.inf]
    choices = []
    for threshold in candidates:
        pred = scores > threshold
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if tp + fn else 0.0
        specificity = tn / (tn + fp) if tn + fp else 0.0
        balanced = (sensitivity + specificity) / 2
        if specificity >= minimum_specificity:
            choices.append((balanced, sensitivity, specificity, float(threshold), (tn, fp, fn, tp)))
    return max(choices, key=lambda item: (item[0], item[1], item[2], item[3]))

methods = ['global_score', 'tile_max_score', 'tile_top2_mean_score']
comparison = []
for method in methods:
    scores = np.asarray([row[method] for row in records], dtype=np.float64)
    auc = float(roc_auc_score(labels, scores))
    ap = float(average_precision_score(labels, scores))
    balanced, sensitivity, specificity, threshold, counts = choose_threshold(labels, scores)
    tn, fp, fn, tp = counts
    comparison.append({
        'method': method,
        'auroc': auc,
        'average_precision': ap,
        'threshold': threshold,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'balanced_accuracy': balanced,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    })

# Select by AUROC first, then balanced accuracy. This choice is frozen for the later final run.
selected = max(comparison, key=lambda row: (row['auroc'], row['balanced_accuracy']))
with (OUTPUT / 'method_comparison.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(comparison[0]))
    writer.writeheader(); writer.writerows(comparison)

selection = {
    'status': 'development_validation_completed',
    'final_test_claim': False,
    'checkpoint_step': int(checkpoint['step']),
    'validation_normal_count': int(np.sum(labels == 0)),
    'validation_stain_count': int(np.sum(labels == 1)),
    'stain_slice': '[100:130] after seed-230224 shuffle; disjoint from V1-V5 [0:100]',
    't_distance': T_DISTANCE,
    'num_diffusion_samples': NUM_DIFFUSION_SAMPLES,
    'tile_source_size': TILE_SOURCE_SIZE,
    'tile_stride': TILE_STRIDE,
    'tile_batch_size': TILE_BATCH_SIZE,
    'tile_pixel_quantile': TILE_PIXEL_QUANTILE,
    'comparison': comparison,
    'selected_method': selected['method'],
    'frozen_threshold': selected['threshold'],
    'run_seconds': run_seconds,
    'peak_gpu_allocated_gib': torch.cuda.max_memory_allocated() / 2**30,
    'next_step': 'Run the selected method and frozen threshold once on a disjoint final stain cohort.',
}
(OUTPUT / 'model_selection.json').write_text(json.dumps(selection, indent=2))
print(json.dumps(selection, indent=2))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, method in zip(axes[0], methods):
    normal = [row[method] for row in records if row['label'] == 0]
    stain = [row[method] for row in records if row['label'] == 1]
    ax.boxplot([normal, stain], labels=['normal', 'stain'], showfliers=True)
    selected_row = next(row for row in comparison if row['method'] == method)
    ax.axhline(selected_row['threshold'], color='red', linestyle='--', label='selected cutoff')
    ax.set_title(f"{method}\nAUROC={selected_row['auroc']:.3f}")
    ax.grid(alpha=.2); ax.legend(fontsize=8)

preview_indices = sorted(preview_cache)[:3]
for ax, index in zip(axes[1], preview_indices):
    result = preview_cache[index]
    maps = result['tile_maps']
    best = int(np.argmax(np.quantile(maps.reshape(len(maps), -1), TILE_PIXEL_QUANTILE, axis=1)))
    ax.imshow(maps[best], cmap='magma')
    ax.set_title(f"{records[index]['kind']} {Path(records[index]['image']).name}\nbest tile response")
    ax.axis('off')

fig.tight_layout()
fig.savefig(OUTPUT / 'validation_summary.png', dpi=160, bbox_inches='tight')
plt.show()

archive = shutil.make_archive('/kaggle/working/fabric_stain_tiled_validation_results', 'zip', OUTPUT)
print('Download:', archive)
print('This output selects a candidate. It is not the final test result.')
